# JXPaint Tutorial 2 — The CustomGNFW pressure profile

JXPaint's profile is the Arnaud (2010) generalized-NFW thermal-SZ profile
(equivalent to hmfast's `ParametricGNFWPressureProfile`). This notebook shows:
1. the 3D gNFW shape and its line-of-sight projection $F(x)$,
2. the geometry $\theta_{500}(M,z)$ and amplitudes $y_0^{\rm param},\,y_0^{\rm Arnaud}$,
3. the **beam-convolved** 2D shape table $y_t(\log\theta,\log\theta_{500})$ used for painting.

In [ ]:
import os, sys, time
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # share the GPU
sys.path.insert(0, os.path.join("..", "src"))   # run from tutorials/
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import jax
print("JAX devices:", jax.devices())

In [ ]:
from jxpaint.profiles.custom_gnfw import CustomGNFWPressureProfile
p = CustomGNFWPressureProfile()
print("gNFW shape params: P0=%.3f c500=%.3f alpha=%.4f beta=%.4f gamma=%.4f B=%.2f"
      % (p.P0, p.c500, p.alpha, p.beta, p.gamma, p.B))

## 1. gNFW 3D shape and its projection $F(x)$

The dimensionless 3D pressure shape is
$P(s) = (c_{500}s)^{-\gamma}\,[1+(c_{500}s)^\alpha]^{(\gamma-\beta)/\alpha}$,
with $s=r/r_{500c}$. The projected (line-of-sight) shape is
$F(x)=P_0\cdot 2\int_0^\infty P(\sqrt{y^2+x^2})\,dy$ with $x=\theta/\theta_{500}$.

In [ ]:
s = np.logspace(-3, 2, 300)
x = np.logspace(-4, 2, 300)
P3d = np.array(p.gnfw(s))
Fx  = np.array(p.F(x))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(s, P3d); ax[0].set_xlabel("s = r / r500c"); ax[0].set_ylabel("P(s)")
ax[0].set_title("3D gNFW pressure shape")
ax[1].loglog(x, Fx); ax[1].set_xlabel(r"x = $\theta/\theta_{500}$")
ax[1].set_ylabel("F(x)"); ax[1].set_title("Projected LOS shape F(x)")
for a in ax: a.grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()
print("shape integral F(0)/P0/2 =", float(p.F(1e-12))/p.P0/2, " (XGPaint: 0.479049273)")

## 2. Geometry and amplitudes vs mass and redshift

$\theta_{500}=\mathrm{angular\_size}(R_{500}/B^{1/3},z)$ sets the angular size;
$y_0^{\rm param}$ and $y_0^{\rm Arnaud}$ are the two central-$y$ normalisations
(parametric scaling relation vs Arnaud profile integral).

In [ ]:
Mgrid = np.logspace(0, 1.5, 50)          # 1e14 .. ~3e15 Msun
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for zz in [0.1, 0.3, 0.7, 1.5]:
    th500 = np.array([float(p.theta500(mm, zz)) for mm in Mgrid])
    ax[0].loglog(Mgrid, np.degrees(th500)*60, label=f"z={zz}")
    y0p = np.array([float(p.y0_param(mm, zz)) for mm in Mgrid])
    y0a = np.array([float(p.y0_arnaud(mm, zz)) for mm in Mgrid])
    ax[1].loglog(Mgrid, y0p, label=f"y0_param z={zz}")
    ax[1].loglog(Mgrid, y0a, "--", label=f"y0_arnaud z={zz}")
ax[0].set_xlabel("M500c [1e14 Msun]"); ax[0].set_ylabel(r"$\theta_{500}$ [arcmin]")
ax[0].set_title("Angular size"); ax[0].legend(); ax[0].grid(alpha=.3, which="both")
ax[1].set_xlabel("M500c [1e14 Msun]"); ax[1].set_ylabel("central y0")
ax[1].set_title("Amplitudes"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

## 3. The beam-convolved 2D shape table

Painting uses a precomputed table $y_t(\log\theta,\log\theta_{500})$: the
projected shape $F(\theta/\theta_{500})$ **convolved with the 10 arcmin beam**.
This table depends only on the gNFW shape and the beam — *not on cosmology* — so
it is built once and reused (see Tutorial 3). Below: slices at several
$\theta_{500}$ showing the beam-smoothed cores, and the full 2D table.

In [ ]:
st = None
from jxpaint.profiles.shape_table import load_beamed_table
st = load_beamed_table()
import jax.numpy as jnp

theta_arcmin = np.logspace(-1, 2.3, 400)
theta_rad = np.radians(theta_arcmin/60)
plt.figure(figsize=(6,4))
for th500_arcmin in [0.5, 1.0, 2.0, 5.0]:
    l5 = np.log(np.radians(th500_arcmin/60))
    yt = np.array(st.evaluate(np.log(theta_rad), np.full_like(theta_rad, l5)))
    plt.loglog(theta_arcmin, yt, label=fr"$\theta_{{500}}$={th500_arcmin}'")
plt.axvline(10, color="k", ls=":", alpha=.5, label="10' beam FWHM")
plt.xlabel(r"$\theta$ [arcmin]"); plt.ylabel(r"$y_t(\theta,\theta_{500})$ / amplitude")
plt.title("Beam-convolved shape, radial slices"); plt.legend(); plt.grid(alpha=.3, which="both")
plt.tight_layout(); plt.show()

In [ ]:
# full 2D table (subsampled) as a heatmap
lt = np.linspace(st.lt_min, st.lt_max, 400)
l5 = np.linspace(st.l5_min, st.l5_max, 400)
LT, L5 = np.meshgrid(lt, l5, indexing="ij")
Z = np.array(st.evaluate(LT.ravel(), L5.ravel())).reshape(LT.shape)
plt.figure(figsize=(6,5))
plt.pcolormesh(np.degrees(np.exp(l5))*60, np.degrees(np.exp(lt))*60,
               np.log10(np.clip(Z, 1e-12, None)), shading="auto")
plt.yscale("log"); plt.xscale("log")
plt.xlabel(r"$\theta_{500}$ [arcmin]"); plt.ylabel(r"$\theta$ [arcmin]")
plt.colorbar(label=r"$\log_{10}\,y_t$"); plt.title("Beam-convolved 2D shape table")
plt.tight_layout(); plt.show()

**Takeaway:** the painted value for a halo is `y0_true / B^(1/3)` times this
beam-convolved table evaluated at $(\theta,\theta_{500})$. Next:
[Tutorial 3](03_speed_and_cosmology.ipynb) — speed and varying cosmology.